# Question B1 (15 marks)

Real world datasets often have a mix of numeric and categorical features – this dataset is one example. To build models on such data, categorical features have to be encoded or embedded.

PyTorch Tabular is a library that makes it very convenient to build neural networks for tabular data. It is built on top of PyTorch Lightning, which abstracts away boilerplate model training code and makes it easy to integrate other tools, e.g. TensorBoard for experiment tracking.

For questions B1 and B2, the following features should be used:   
- **Numeric / Continuous** features: dist_to_nearest_stn, dist_to_dhoby, degree_centrality, eigenvector_centrality, remaining_lease_years, floor_area_sqm
- **Categorical** features: month, town, flat_model_type, storey_range



---



In [3]:
!pip install "pytorch_tabular[extra]"

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 9.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/65.9 MB ? eta -:--:--
    --------------------------------------- 1.6/65.9 MB 8.4 MB/s eta 0:00:08
   - -------------------------------------- 3.1/65.9 MB 7.7 MB/s eta 0:00:09
   --- ------------------------------------ 5.2/65.9 MB 8.4 MB/s eta 0:00:08
   ---- ----------------------------------- 7.3/65.9 MB 8.9 MB/s eta 0:00:07
   ----- ---------------------------------- 8.9/65.9 MB 8.5 MB/s eta 0:00:07
   ------ --------------------------------- 10.0/65.9 MB 8.1 MB/s eta 0:00:07
   ------ --------------------------------- 11.5/65.9 MB 8.0 MB/s eta 0:00:07
   ------- -------------------------------- 13.1/65.9 MB 7.8 MB/s eta 0:00:07
   --------- ------------------------------ 15.2/65.9 MB 8.1 MB/

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 26.0 which is incompatible.


In [1]:
SEED = 42

import os

import random
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

import pandas as pd

from pytorch_tabular import TabularModel
from pytorch_tabular.models import CategoryEmbeddingModelConfig
from pytorch_tabular.config import (
    DataConfig,
    OptimizerConfig,
    TrainerConfig,
)

from sklearn.metrics import mean_squared_error, r2_score

1.Divide the dataset (‘hdb_price_prediction.csv’) into train, validation and test sets by using entries from year 2019 and before as training data, year 2020 as validation data and year 2021 as test data.
**Do not** use data from year 2022 and year 2023.



In [6]:
df = pd.read_csv('hdb_price_prediction.csv')

# YOUR CODE HERE

# Filter Out data from year 2022 and 2023
df = df[df['year'].isin([2019, 2020, 2021]) | (df['year'] < 2019)]

# Define features that should be used
numeric_features     = ['dist_to_nearest_stn', 'dist_to_dhoby', 'degree_centrality', 'eigenvector_centrality', 'remaining_lease_years', 'floor_area_sqm']
categorical_features = ['month', 'town', 'flat_model_type', 'storey_range']
prediction_target    = ['resale_price']
all_features         = numeric_features + categorical_features + prediction_target

# Split features for train, validation & test by year
train_df = df[df['year'] <= 2019]
val_df   = df[df['year'] == 2020]
test_df  = df[df['year'] == 2021]

# Separate features and target
train_df = train_df[all_features]
val_df   = val_df[all_features]
test_df  = test_df[all_features]

In [7]:
display(train_df)

,dist_to_nearest_stn,dist_to_dhoby,degree_centrality,eigenvector_centrality,remaining_lease_years,floor_area_sqm,month,town,flat_model_type,storey_range,resale_price
0,1.007264,7.006044,0.016807,0.006243,61.333333,44.0,1,ANG MO KIO,"2 ROOM, Improved",10 TO 12,232000.0
1,1.271389,7.983837,0.016807,0.006243,60.583333,67.0,1,ANG MO KIO,"3 ROOM, New Generation",01 TO 03,250000.0
2,1.069743,9.090700,0.016807,0.002459,62.416667,67.0,1,ANG MO KIO,"3 ROOM, New Generation",01 TO 03,262000.0
3,0.946890,7.519889,0.016807,0.006243,62.083333,68.0,1,ANG MO KIO,"3 ROOM, New Generation",04 TO 06,265000.0
4,1.092551,9.130489,0.016807,0.002459,62.416667,67.0,1,ANG MO KIO,"3 ROOM, New Generation",01 TO 03,265000.0
...,...,...,...,...,...,...,...,...,...,...,...
64052,0.823163,14.421823,0.016807,0.000382,67.583333,142.0,12,YISHUN,"EXECUTIVE, Apartment",04 TO 06,580000.0
64053,0.823163,14.421823,0.016807,0.000382,67.583333,146.0,12,YISHUN,"EXECUTIVE, Maisonette",07 TO 09,565000.0
64054,0.445869,13.498243,0.016807,0.000968,71.500000,164.0,12,YISHUN,"EXECUTIVE, Apartment",01 TO 03,633000.0
64055,0.552769,13.598257,0.016807,0.000968,71.500000,164.0,12,YISHUN,"EXECUTIVE, Apartment",10 TO 12,788888.0


In [10]:
display(val_df)

,dist_to_nearest_stn,dist_to_dhoby,degree_centrality,eigenvector_centrality,remaining_lease_years,floor_area_sqm,month,town,flat_model_type,storey_range,resale_price
64057,0.917344,7.336493,0.016807,0.006243,55.583333,73.0,1,ANG MO KIO,"3 ROOM, New Generation",04 TO 06,265000.0
64058,0.696776,7.341622,0.016807,0.006243,91.666667,70.0,1,ANG MO KIO,"3 ROOM, Model A",19 TO 21,470000.0
64059,0.597608,7.292217,0.016807,0.006243,56.333333,73.0,1,ANG MO KIO,"3 ROOM, New Generation",01 TO 03,230000.0
64060,0.994153,7.427003,0.016807,0.006243,55.250000,73.0,1,ANG MO KIO,"3 ROOM, New Generation",04 TO 06,280000.0
64061,0.921541,8.163605,0.016807,0.006243,59.083333,68.0,1,ANG MO KIO,"3 ROOM, New Generation",07 TO 09,220000.0
...,...,...,...,...,...,...,...,...,...,...,...
87365,1.153544,14.075870,0.016807,0.000382,66.666667,146.0,12,YISHUN,"EXECUTIVE, Maisonette",04 TO 06,560000.0
87366,1.254784,13.948192,0.016807,0.000382,66.750000,145.0,12,YISHUN,"EXECUTIVE, Apartment",01 TO 03,540000.0
87367,0.466763,13.426086,0.016807,0.000968,66.000000,142.0,12,YISHUN,"EXECUTIVE, Apartment",13 TO 15,638000.0
87368,0.281375,12.884815,0.016807,0.000968,66.166667,146.0,12,YISHUN,"EXECUTIVE, Maisonette",10 TO 12,683500.0


In [12]:
display(test_df)

,dist_to_nearest_stn,dist_to_dhoby,degree_centrality,eigenvector_centrality,remaining_lease_years,floor_area_sqm,month,town,flat_model_type,storey_range,resale_price
87370,1.276775,8.339960,0.016807,0.002459,64.083333,45.0,1,ANG MO KIO,"2 ROOM, Improved",01 TO 03,211000.0
87371,1.276775,8.339960,0.016807,0.002459,64.083333,45.0,1,ANG MO KIO,"2 ROOM, Improved",07 TO 09,225000.0
87372,0.884872,6.981730,0.016807,0.006243,59.000000,68.0,1,ANG MO KIO,"3 ROOM, New Generation",04 TO 06,260000.0
87373,0.677246,8.333056,0.016807,0.006243,58.166667,68.0,1,ANG MO KIO,"3 ROOM, New Generation",04 TO 06,265000.0
87374,0.922047,8.009223,0.016807,0.006243,58.083333,68.0,1,ANG MO KIO,"3 ROOM, New Generation",01 TO 03,265000.0
...,...,...,...,...,...,...,...,...,...,...,...
116422,0.954699,13.018048,0.016807,0.000968,95.083333,112.0,12,YISHUN,"5 ROOM, Improved",13 TO 15,720000.0
116423,0.475885,12.738721,0.016807,0.000968,65.083333,142.0,12,YISHUN,"EXECUTIVE, Apartment",01 TO 03,738000.0
116424,0.408137,12.745325,0.016807,0.000968,65.000000,146.0,12,YISHUN,"EXECUTIVE, Maisonette",04 TO 06,755000.0
116425,0.733238,14.183095,0.016807,0.000382,90.916667,112.0,12,YISHUN,"5 ROOM, DBSS",10 TO 12,848000.0


In [14]:
# Make sure data sets have no null values
print(train_df.isnull().sum().sum())
print(val_df.isnull().sum().sum())
print(test_df.isnull().sum().sum())

0
0
0


2.Refer to the documentation of **PyTorch Tabular** and perform the following tasks: https://pytorch-tabular.readthedocs.io/en/latest/#usage
- Use **[DataConfig](https://pytorch-tabular.readthedocs.io/en/latest/data/)** to define the target variable, as well as the names of the continuous and categorical variables.
- Use **[TrainerConfig](https://pytorch-tabular.readthedocs.io/en/latest/training/)** to automatically tune the learning rate. Set batch_size to be 1024 and set max_epoch as 50.
- Use **[CategoryEmbeddingModelConfig](https://pytorch-tabular.readthedocs.io/en/latest/models/#category-embedding-model)** to create a feedforward neural network with 1 hidden layer containing 50 neurons.
- Use **[OptimizerConfig](https://pytorch-tabular.readthedocs.io/en/latest/optimizer/)** to choose Adam optimiser. There is no need to set the learning rate (since it will be tuned automatically) nor scheduler.
- Use **[TabularModel](https://pytorch-tabular.readthedocs.io/en/latest/tabular_model/)** to initialise the model and put all the configs together.

In [17]:
batch_size = 1024
max_epochs = 50
layers = '50'
optimizer = 'Adam'

In [19]:
# YOUR CODE HERE

data_config = DataConfig(
    target = prediction_target, 
    continuous_cols = numeric_features, 
    categorical_cols = categorical_features
)

trainer_config = TrainerConfig(
    auto_lr_find = True, 
    batch_size = batch_size, 
    max_epochs = max_epochs
)

model_config = CategoryEmbeddingModelConfig(
    task = 'regression', 
    layers = layers, 
)

optimizer_config = OptimizerConfig(
    optimizer = optimizer
)

tabular_model = TabularModel(
    data_config = data_config, 
    model_config = model_config, 
    optimizer_config = optimizer_config, 
    trainer_config = trainer_config
)

2026-03-09 20:12:13,123 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off


In [21]:
# Train the model
tabular_model.fit(train = train_df, validation = val_df)

Seed set to 42
2026-03-09 20:12:16,408 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-03-09 20:12:16,473 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for regression task
2026-03-09 20:12:16,713 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: CategoryEmbeddingModel
2026-03-09 20:12:16,830 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
C:\Users\chooz\anaconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one 

Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=100` reached.
Restoring states from the checkpoint path at C:\Users\chooz\OneDrive\Documents\SC4001 Neural Network\SC4001 Assignment\.lr_find_3bd6074e-5496-465b-842e-b7286146f093.ckpt
C:\Users\chooz\anaconda3\Lib\site-packages\lightning_fabric\utilities\cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=T

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ CategoryEmbeddingBackbone │  2.9 K │ train │     0 │
│ 1 │ _embedding_layer │ Embedding1dLayer          │  1.5 K │ train │     0 │
│ 2 │ head             │ LinearHead                │     51 │ train │     0 │
│ 3 │ loss             │ MSELoss                   │      0 │ train │     0 │
└───┴──────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 4.5 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.5 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 16                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=50` reached.


2026-03-09 20:17:41,005 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-03-09 20:17:41,007 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model
C:\Users\chooz\anaconda3\Lib\site-packages\pytorch_tabular\utils\python_utils.py:92: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for 

3.Report the test MSE error and the test R2 value that you obtained.



In [23]:
# YOUR CODE & RESULT HERE
evaluation = tabular_model.evaluate(test_df)
pred_df = tabular_model.predict(test_df)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

C:\Users\chooz\anaconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │      287991529472.0       │
│  test_mean_squared_error  │      287991529472.0       │
└───────────────────────────┴───────────────────────────┘

In [26]:
test_values  = test_df['resale_price'].values
pred_values  = pred_df['resale_price_prediction'].values

# Calculate MSE and R2
test_mse = mean_squared_error(test_values, pred_values)
test_r2  = r2_score(test_values, pred_values)

print(f"Test MSE: {test_mse:.4f}")
print(f"Test R2:  {test_r2:.4f}")

Test MSE: 287991537572.3084
Test R2:  -9.8873


4.Print out the corresponding rows in the dataframe for the top 20 test samples with the largest errors. Are there any patterns or common characteristics among these data points? Based on your observations, suggest and briefly explain a potential strategy to improve the model on these cases.



In [44]:
# YOUR CODE & RESULT HERE

errors = np.abs(test_values - pred_values)

# Get top 20 largest errors
top20_indices = np.argsort(errors)[-20:][::-1]
top20_df = test_df.iloc[top20_indices].copy()
top20_df['predicted_price'] = pred_values[top20_indices]
top20_df['absolute_error'] = errors[top20_indices]

print("Top 20 Test Samples with the Largest Errors:")
display(top20_df.sort_values('absolute_error', ascending = False))

Top 20 Test Samples with the Largest Errors:


,dist_to_nearest_stn,dist_to_dhoby,degree_centrality,eigenvector_centrality,remaining_lease_years,floor_area_sqm,month,town,flat_model_type,storey_range,resale_price,predicted_price,absolute_error
90608,0.776182,6.297489,0.033613,0.015854,88.833333,120.0,12,BISHAN,"5 ROOM, DBSS",37 TO 39,1360000.0,2.891364,1.359997e+06
106199,0.584731,3.882019,0.016807,0.008342,93.333333,122.0,12,QUEENSTOWN,"5 ROOM, Premium Apartment Loft",40 TO 42,1328000.0,4.643356,1.327995e+06
90483,0.767244,6.327956,0.033613,0.015854,89.000000,120.0,9,BISHAN,"5 ROOM, DBSS",37 TO 39,1295000.0,4.889019,1.294995e+06
93931,0.352779,2.413099,0.033613,0.121082,88.083333,107.0,12,CENTRAL AREA,"5 ROOM, Type S2",40 TO 42,1288000.0,5.689346,1.287994e+06
93930,0.438348,2.506568,0.033613,0.121082,88.166667,107.0,12,CENTRAL AREA,"5 ROOM, Type S2",46 TO 48,1280000.0,6.224119,1.279994e+06
90432,0.827889,6.370404,0.033613,0.015854,88.916667,120.0,8,BISHAN,"5 ROOM, DBSS",25 TO 27,1280000.0,6.917776,1.279993e+06
100836,0.998313,3.304953,0.016807,0.053004,50.083333,210.0,6,KALLANG/WHAMPOA,"3 ROOM, Terrace",01 TO 03,1268000.0,1.831578,1.267998e+06
101237,0.352251,2.587444,0.016807,0.004414,88.250000,119.0,11,KALLANG/WHAMPOA,"5 ROOM, DBSS",40 TO 42,1268000.0,2.434384,1.267998e+06
93904,0.401367,2.445314,0.033613,0.121082,88.333333,106.0,11,CENTRAL AREA,"5 ROOM, Type S2",40 TO 42,1261000.0,6.128878,1.260994e+06
90523,0.776182,6.297489,0.033613,0.015854,88.916667,120.0,10,BISHAN,"5 ROOM, DBSS",22 TO 24,1260000.0,7.799955,1.259992e+06
